# Multi-Segment Batch Summary

This notebook summarizes batched `eval_vlm_multisegment.py` results for one model.

It builds:
- a long-form config summary table
- `fps x caption x gaze` matrix tables for hit rate and core VLM metrics
- optional representative-frame tables when `refined_result_multiseg_*.json` files exist


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RESULTS_DIR = Path("results")
MODEL_NAME = "gemini-3.1-flash-lite-preview"


In [2]:
def load_json(path: Path) -> dict | None:
    if not path.exists():
        return None
    return json.loads(path.read_text())


def collect_result_paths(results_dir: Path, model_name: str) -> list[Path]:
    paths = sorted(results_dir.glob("result_multiseg_*.json"))
    filtered = []
    for path in paths:
        payload = load_json(path)
        if not isinstance(payload, dict):
            continue
        config = payload.get("config") or {}
        if config.get("model") == model_name:
            filtered.append(path)
    return filtered


def get_nested(data: dict, *keys, default=None):
    cur = data
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
    return default if cur is None else cur


def load_summary_rows(results_dir: Path, model_name: str) -> list[dict]:
    rows = []
    for result_path in collect_result_paths(results_dir, model_name):
        payload = load_json(result_path)
        if payload is None:
            continue

        config = payload.get("config") or {}
        agg = payload.get("aggregate") or {}

        row = {
            "file": result_path.name,
            "fps": int(config.get("video_fps")),
            "caption": bool(config.get("caption")),
            "gaze_annot": bool(config.get("gaze_annot")),
            "num_episodes": get_nested(agg, "num_episodes"),
            "num_success": get_nested(agg, "num_success"),
            "count_acc": get_nested(agg, "exact_count_accuracy", "rate"),
            "all_gt_matched": get_nested(agg, "all_gt_matched_rate", "rate"),
            "all_pred_matched": get_nested(agg, "all_pred_matched_rate", "rate"),
            "seg_f1": get_nested(agg, "segment_f1", "mean"),
            "frame_iou": get_nested(agg, "frame_iou", "mean"),
            "pred_mid_hit": get_nested(agg, "pred_midpoint_hit_rate", "mean"),
            "pred_mid_nearest": get_nested(agg, "pred_midpoint_nearest_dist_mean", "mean"),
        }

        refined_path = results_dir / f"refined_{result_path.name}"
        refined_payload = load_json(refined_path)
        refined_agg = refined_payload.get("aggregate") if isinstance(refined_payload, dict) else {}
        row.update(
            {
                "refined_file": refined_path.name if refined_path.exists() else "",
                "mid_exact": get_nested(refined_agg, "midpoint", "exact_accuracy"),
                "gaze_exact": get_nested(refined_agg, "gaze_velocity", "exact_accuracy"),
                "mid_hit_at_1": get_nested(refined_agg, "midpoint", "hit@1_rate"),
                "gaze_hit_at_1": get_nested(refined_agg, "gaze_velocity", "hit@1_rate"),
                "mid_all_ep": get_nested(refined_agg, "midpoint", "all_interval_episode_rate"),
                "gaze_all_ep": get_nested(refined_agg, "gaze_velocity", "all_interval_episode_rate"),
                "mid_ord_gt": get_nested(refined_agg, "midpoint", "ordered_gt_hit_recall"),
                "gaze_ord_gt": get_nested(refined_agg, "gaze_velocity", "ordered_gt_hit_recall"),
            }
        )
        rows.append(row)
    return rows


In [3]:
summary_df = pd.DataFrame(load_summary_rows(RESULTS_DIR, MODEL_NAME)).sort_values(
    ["fps", "caption", "gaze_annot"]
).reset_index(drop=True)

if summary_df.empty:
    raise ValueError(f"No result_multiseg files found for model={MODEL_NAME}")

summary_df["caption_label"] = summary_df["caption"].map({False: "w/o live caption", True: "w/ live caption"})
summary_df["gaze_label"] = summary_df["gaze_annot"].map({False: "w/o gaze annotation", True: "w/ gaze annotation"})

display(Markdown(f"## Raw Config Summary: `{MODEL_NAME}`"))
display(
    summary_df[
        [
            "fps",
            "caption",
            "gaze_annot",
            "count_acc",
            "all_gt_matched",
            "all_pred_matched",
            "seg_f1",
            "frame_iou",
            "pred_mid_hit",
            "pred_mid_nearest",
            "file",
            "refined_file",
        ]
    ].round(4)
)


## Raw Config Summary: `gemini-3.1-flash-lite-preview`

,fps,caption,gaze_annot,count_acc,all_gt_matched,all_pred_matched,seg_f1,frame_iou,pred_mid_hit,pred_mid_nearest,file,refined_file
0,2,False,False,0.8493,0.7534,0.8630,0.8858,0.3710,0.5479,3.5342,result_multiseg_gemini-3.1-flash-lite-preview_...,
1,2,False,True,0.7222,0.5972,0.8056,0.8032,0.2940,0.3819,7.0069,result_multiseg_gemini-3.1-flash-lite-preview_...,
2,2,True,False,0.9155,0.8873,0.9437,0.9507,0.3743,0.5000,2.4014,result_multiseg_gemini-3.1-flash-lite-preview_...,
3,2,True,True,0.7671,0.6986,0.8630,0.8402,0.3298,0.4863,6.1986,result_multiseg_gemini-3.1-flash-lite-preview_...,
4,4,False,False,0.9178,0.7260,0.7808,0.8630,0.3709,0.5959,4.5822,result_multiseg_gemini-3.1-flash-lite-preview_...,refined_result_multiseg_gemini-3.1-flash-lite-...
5,4,False,True,0.8028,0.5915,0.7183,0.7653,0.2990,0.3873,10.1761,result_multiseg_gemini-3.1-flash-lite-preview_...,
6,4,True,False,0.9403,0.8806,0.8955,0.9030,0.3895,0.5970,5.3582,result_multiseg_gemini-3.1-flash-lite-preview_...,
7,4,True,True,0.9041,0.7945,0.8493,0.8836,0.3247,0.4384,4.5753,result_multiseg_gemini-3.1-flash-lite-preview_...,


In [4]:
ROW_ORDER = ["w/o live caption", "w/ live caption"]
COL_ORDER = ["w/o gaze annotation", "w/ gaze annotation"]


def build_metric_matrix(df: pd.DataFrame, fps: int, metric: str) -> pd.DataFrame:
    subset = df[df["fps"] == fps].copy()
    matrix = subset.pivot(index="caption_label", columns="gaze_label", values=metric)
    matrix = matrix.reindex(index=ROW_ORDER, columns=COL_ORDER)
    return matrix


def display_metric_tables(df: pd.DataFrame, metric: str, title: str, percent: bool = True, digits: int = 1) -> None:
    display(Markdown(f"## {title}"))
    for fps in sorted(df["fps"].dropna().unique()):
        matrix = build_metric_matrix(df, int(fps), metric)
        style = matrix.style
        if percent:
            style = style.format(lambda value: "-" if pd.isna(value) else f"{value * 100:.1f}%")
        else:
            style = style.format(lambda value: "-" if pd.isna(value) else f"{value:.{digits}f}")
        style = style.set_caption(f"fps = {int(fps)}").highlight_max(axis=None, color="#dff6dd")
        display(style)


In [5]:
display_metric_tables(summary_df, "pred_mid_hit", f"VLM: {MODEL_NAME} / midpoint hit rate", percent=True)
display_metric_tables(summary_df, "count_acc", "VLM / exact count match", percent=True)
display_metric_tables(summary_df, "seg_f1", "VLM / segment F1", percent=False, digits=3)
display_metric_tables(summary_df, "frame_iou", "VLM / frame IoU", percent=False, digits=3)


## VLM: gemini-3.1-flash-lite-preview / midpoint hit rate

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,54.8%,38.2%
w/ live caption,50.0%,48.6%


gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,59.6%,38.7%
w/ live caption,59.7%,43.8%


## VLM / exact count match

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,84.9%,72.2%
w/ live caption,91.5%,76.7%


...

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,0.886,0.803
w/ live caption,0.951,0.840


gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,0.863,0.765
w/ live caption,0.903,0.884


## VLM / frame IoU

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,0.371,0.294
w/ live caption,0.374,0.330


gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,0.371,0.299
w/ live caption,0.390,0.325


In [6]:
refined_df = summary_df[summary_df["mid_exact"].notna() | summary_df["gaze_exact"].notna()].copy()

if refined_df.empty:
    display(Markdown("## Representative Frame Tables\nNo `refined_result_multiseg_*.json` files were found for this model."))
else:
    display_metric_tables(refined_df, "mid_exact", f"Representative frame: {MODEL_NAME} / midpoint exact hit", percent=True)
    display_metric_tables(refined_df, "gaze_exact", f"Representative frame: {MODEL_NAME} / min gaze-speed exact hit", percent=True)
    display_metric_tables(refined_df, "mid_all_ep", "Representative frame / midpoint all-interval episode hit", percent=True)
    display_metric_tables(refined_df, "gaze_all_ep", "Representative frame / min gaze-speed all-interval episode hit", percent=True)


## Representative frame: gemini-3.1-flash-lite-preview / midpoint exact hit

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,61.0%,-
w/ live caption,-,-


## Representative frame: gemini-3.1-flash-lite-preview / min gaze-speed exact hit

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,43.3%,-
w/ live caption,-,-


## Representative frame / midpoint all-interval episode hit

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,41.1%,-
w/ live caption,-,-


## Representative frame / min gaze-speed all-interval episode hit

gaze_label,w/o gaze annotation,w/ gaze annotation
caption_label,,
w/o live caption,20.5%,-
w/ live caption,-,-


In [7]:
output_dir = Path("debug/multiseg_summary")
output_dir.mkdir(parents=True, exist_ok=True)

summary_csv_path = output_dir / f"{MODEL_NAME}_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

for metric_name in ["pred_mid_hit", "count_acc", "seg_f1", "frame_iou"]:
    for fps in sorted(summary_df["fps"].dropna().unique()):
        matrix = build_metric_matrix(summary_df, int(fps), metric_name)
        matrix.to_csv(output_dir / f"{MODEL_NAME}_{metric_name}_fps{int(fps)}.csv")

print(f"Saved summary CSV to: {summary_csv_path}")
print(f"Saved matrix CSV files under: {output_dir}")


Saved summary CSV to: debug/multiseg_summary/gemini-3.1-flash-lite-preview_summary.csv
Saved matrix CSV files under: debug/multiseg_summary
